<a href="https://colab.research.google.com/github/sunnybal2010/codespaces-blank/blob/main/Heart_Attack_Statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
pip install nhanes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 12.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import pandas as pd
from nhanes.load import load_NHANES_data
import os
import re

data_frame = load_NHANES_data(year = '2017-2018')


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import log_loss
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_selection import VarianceThreshold
import numpy as np
import pandas as pd

target = 'EverToldYouHadHeartAttack'

data_clean = data_frame[data_frame[target].isin([0, 1, 0.0, .0])].copy()

X = data_clean.drop([target], axis=1)
y = data_clean[target].astype(int)

X_num = pd.get_dummies(X, drop_first=True)

cols_with_all_nan = X_num.columns[X_num.isnull().all()]
if not cols_with_all_nan.empty:
    X_num = X_num.drop(columns=cols_with_all_nan)

clinical_cols = [
    'AgeInYearsAtScreening',
    'SystolicBloodPres1StRdgMmHg',
    'DiastolicBloodPres1StRdgMmHg',
    'TotalCholesterolMgdl',
    'DirectHdlcholesterolMgdl',
    'BpTreated',
    'SmokedAtLeast100CigarettesInLife',
    'DoctorToldYouHaveDiabetes_1',
    'BodyMassIndexKgm2',
    'WaistCircumferenceCm',
]


clinical_cols = [col for col in clinical_cols if col in X_num.columns]
X_num = X_num[clinical_cols]

X_num['PulsePressure'] = X_num['SystolicBloodPres1StRdgMmHg'] - X_num['DiastolicBloodPres1StRdgMmHg']
X_num['NonHDL'] = X_num['TotalCholesterolMgdl'] - X_num['DirectHdlcholesterolMgdl']
X_num['Age_x_SBP'] = X_num['AgeInYearsAtScreening'] * X_num['SystolicBloodPres1StRdgMmHg']

print(X_num.shape)

X_train, X_test, y_train, y_test = train_test_split(X_num, y, test_size=0.2, random_state=42)


pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(interaction_only=True, include_bias=False)),
    ('selector', SelectKBest(f_classif, k=20)),
    ('logistic', LogisticRegression(max_iter=1000, solver='saga'))
])

p_grid = {
    'poly__degree': [1, 2],
    'selector__k': [20, 30, 40],
    'logistic__C': [0.001, 0.01, 0.1, 1, 5],
    'logistic__penalty': ['l1', 'l2'],
    'logistic__class_weight': ['balanced', None]
}

print(X_num.shape)
cv_fold = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
grid_search = GridSearchCV(pipeline, p_grid, cv=cv_fold, scoring='neg_log_loss', n_jobs=-1, verbose=2)
grid_search.fit(X_train.values, y_train.values)

model = grid_search.best_estimator_
print(f"Best Parameters: {grid_search.best_params_}")

probs = model.predict_proba(X_test.values)
current_log_loss = log_loss(y_test, probs)

null_probs = np.full(y_test.shape, y_test.mean())
null_log_loss = log_loss(y_test, null_probs)

pseudo_r2 = 1 - (current_log_loss / null_log_loss)
print(f"McFadden pseudo R-Squared {pseudo_r2:.4f}")


(5255, 12)
(5255, 12)
Fitting 15 folds for each of 120 candidates, totalling 1800 fits
Best Parameters: {'logistic__C': 0.1, 'logistic__class_weight': None, 'logistic__penalty': 'l1', 'poly__degree': 1, 'selector__k': 30}
McFadden pseudo R-Squared 0.1308


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:783: UserWarning: k=30 is greater than n_features=12. All the features will be returned.
  warnings.warn(


In [ ]:

def get_user_input():
    print("\n--- Heart Attack Risk Assessment ---")
    data = {
        'AgeInYearsAtScreening': float(input("Age: ")),
        'Gender_Male': 1 if input("Gender (M/F): ").lower() == 'm' else 0,
        'SystolicBloodPres1StRdgMmHg': float(input("Systolic Blood Pressure (e.g. 120): ")),
        'TotalCholesterolMgdl': float(input("Total Cholesterol (e.g. 200): ")),
        'DirectHdlcholesterolMgdl': float(input("HDL Cholesterol (e.g. 50): ")),
        'BpTreated': 1 if input("Are you on blood pressure medication? (y/n): ").lower() == 'y' else 0,
        'SmokedAtLeast100CigarettesInLife': 1 if input("Do you smoke? (y/n): ").lower() == 'y' else 0,
        'DoctorToldYouHaveDiabetes_1': 1 if input("Do you have diabetes? (y/n): ").lower() == 'y' else 0
    }
    return data


def ascvd_1year_risk(age, is_male, systolic_bp, total_cholesterol, hdl_cholesterol, is_smoker, is_diabetic, bp_treated):
    ln_age = np.log(age)
    ln_tc  = np.log(total_cholesterol)

    if is_male:
        sbp_term  = -1.799 * np.log(systolic_bp) if bp_treated else -7.990 * np.log(systolic_bp)
        mean_coef = 47.2908 if bp_treated else 17.6515
        sum_coeff = (12.344 * ln_age
                   + 11.853 * ln_tc
                   -  2.664 * ln_age * ln_tc
                   + sbp_term
                   +  1.769 * is_smoker
                   +  0.658 * is_diabetic)
        baseline  = 0.9402
    else:
        sbp_term  = 2.887 * np.log(systolic_bp) if bp_treated else -1.172 * np.log(systolic_bp)
        mean_coef = 14.1362 if bp_treated else -5.2962
        sum_coeff = (-7.574  * ln_age
                   +  1.764  * ln_age**2
                   +  1.794  * ln_tc
                   -  0.317  * ln_age * ln_tc
                   + sbp_term
                   +  0.554  * is_smoker
                   +  0.661  * is_diabetic)
        baseline  = 0.9665

    ten_year_risk = 1 - baseline ** np.exp(sum_coeff - mean_coef)
    ten_year_risk = max(0.0, min(ten_year_risk, 1.0))
    one_year_risk = 1 - (1 - ten_year_risk) ** (1/10)
    return one_year_risk * 100

risk = ascvd_1year_risk(
    age=user_data['AgeInYearsAtScreening'],
    is_male=user_data['Gender_Male'],
    systolic_bp=user_data['SystolicBloodPres1StRdgMmHg'],
    total_cholesterol=user_data['TotalCholesterolMgdl'],
    hdl_cholesterol=user_data['DirectHdlcholesterolMgdl'],
    is_smoker=user_data['SmokedAtLeast100CigarettesInLife'],
    is_diabetic=user_data['DoctorToldYouHaveDiabetes_1'],
    bp_treated=user_data['BpTreated']
)

user_data = get_user_input()


input_df = pd.DataFrame([user_data])
input_df = input_df.reindex(columns=X_num.columns, fill_value=0)
input_scaled = scaler.transform(input_df)

model_prob = model.predict_proba(input_scaled)[0][1] * 100
framingham_prob = ascvd_1year_risk(
    age=user_data['AgeInYearsAtScreening'],
    is_male=user_data['Gender_Male'],
    systolic_bp=user_data['SystolicBloodPres1StRdgMmHg'],
    total_cholesterol=user_data['TotalCholesterolMgdl'],
    hdl_cholesterol=user_data['DirectHdlcholesterolMgdl'],
    is_smoker=user_data['SmokedAtLeast100CigarettesInLife'],
    is_diabetic=user_data['DoctorToldYouHaveDiabetes_1'],
    bp_treated=user_data['BpTreated']
)

risk = (model_prob + framingham_prob) / 2
print(f"\n--- Result ---")
print(f"Estimated 1-year heart attack risk: {risk:.2f}%")

if risk > 1.5:
    print("Warning: This represents a high-risk clinical profile for the coming year.")



--- Heart Attack Risk Assessment ---
Age: 15
Gender (M/F): F
Systolic Blood Pressure (e.g. 120): 110
Total Cholesterol (e.g. 200): 140
HDL Cholesterol (e.g. 50): 50
Are you on blood pressure medication? (y/n): n
Do you smoke? (y/n): n
Do you have diabetes? (y/n): n

--- Result ---
Estimated 1-year heart attack risk: 0.62%


0.543068033359162
0.5430889684620355
0.01386801111676883
0.01386801111676883
3.7302626478491585
